# Standalone Colab Train: BBDM PNG Virtual Staining

Notebook này tự chứa code project bên trong. Bạn chỉ cần upload file `.ipynb` này lên Google Colab, không cần GitHub và không cần upload repo code riêng.

Notebook hiện dùng đúng 2 split dữ liệu:

- `train/input` và `train/target` để train model
- `test/input` và `test/target` để validation/visualize

Ảnh điều kiện được đọc là grayscale. Ảnh `Condition` hiển thị trong notebook là raw grayscale input, không phải output màu của encoder.


In [ ]:
# Check GPU. In Colab: Runtime -> Change runtime type -> GPU.
!nvidia-smi


## 1. Install Dependencies

Cell này cài OpenMPI cho `mpi4py`, sau đó cài các package Python cần cho project.


In [ ]:
!apt-get -qq update
!apt-get -qq install -y libopenmpi-dev openmpi-bin
%pip -q install blobfile mpi4py pytorch-wavelets lpips scikit-image opencv-python tqdm datasets


## 2. Extract Embedded Project Code

Không cần clone GitHub. Code project đã được nhúng trong notebook này và sẽ được giải nén vào `/content/Super-resolved-virtual-staining`.


In [ ]:
import base64
import io
import os
import shutil
import sys
import zipfile
from pathlib import Path

PROJECT_ZIP_B64 = '''
...
'''

WORKDIR = Path('/content/Super-resolved-virtual-staining')
if WORKDIR.exists():
    shutil.rmtree(WORKDIR)
WORKDIR.mkdir(parents=True, exist_ok=True)

zip_bytes = base64.b64decode(PROJECT_ZIP_B64.encode('ascii'))
with zipfile.ZipFile(io.BytesIO(zip_bytes)) as zf:
    zf.extractall(WORKDIR)

os.chdir(WORKDIR)
sys.path.insert(0, str(WORKDIR))

print('Project extracted to:', WORKDIR)
print('Files:', sorted(p.name for p in WORKDIR.iterdir())[:20])


## 3. Locate Dataset Source

Mặc định notebook tải dataset trực tiếp từ Hugging Face rồi convert về cấu trúc split như sau:

- `train/input`: 4800 ảnh grayscale
- `train/target`: 4800 ảnh RGB
- `test/input`: 1200 ảnh grayscale
- `test/target`: 1200 ảnh RGB

Nếu muốn quay lại dùng data trên Drive hoặc upload zip thủ công, đổi `DATA_SOURCE`.


In [ ]:
from pathlib import Path
import shutil
import zipfile

# Options: 'huggingface', 'drive', or 'upload'.
DATA_SOURCE = 'huggingface'
HF_DATASET_ID = 'mezeidragos-lateral/bach-breast-histopathology-he-staining-patches-512x512'
HF_INPUT_COLUMN = 'unstained_image'
HF_TARGET_COLUMN = 'stained_image'
HF_EXPORT_PREPARED_ZIP_TO_DRIVE = False  # Optional cache; keep False to avoid slow Drive writes.

# Drive is still useful for checkpoint/output persistence, even when data comes from Hugging Face.
SAVE_CHECKPOINTS_TO_DRIVE = True
DRIVE_PROJECT_DIR = Path('/content/drive/MyDrive/bach_dataset_download')

# Used only when DATA_SOURCE = 'drive'. Leave None to auto-detect inside DRIVE_PROJECT_DIR.
DRIVE_DATASET_ZIP = None
DRIVE_DATASET_DIR = DRIVE_PROJECT_DIR / 'data'

DATASET_ZIP = None
DATASET_SOURCE_DIR = None
DATASET_HF_ID = None
DRIVE_RUN_ROOT = None
OUTPUT_DRIVE_DIR = Path('/content/bbdm_outputs_standalone')


def has_split_root(path):
    path = Path(path)
    return all((path / split / subdir).exists() for split in ['train', 'test'] for subdir in ['input', 'target'])


def find_split_root(root):
    root = Path(root)
    candidates = [root, root / 'data', root / 'dataset']
    if root.exists():
        for split_name in ['train', 'test']:
            candidates.extend(p.parent for p in root.rglob(split_name) if p.is_dir())
    seen = set()
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        if has_split_root(candidate):
            return candidate
    return None


def resolve_optional_path(value):
    if value is None or value == '':
        return None
    value = Path(value)
    if value.is_absolute():
        return value
    return DRIVE_PROJECT_DIR / value


if SAVE_CHECKPOINTS_TO_DRIVE or DATA_SOURCE == 'drive':
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_PROJECT_DIR.mkdir(parents=True, exist_ok=True)
    DRIVE_RUN_ROOT = DRIVE_PROJECT_DIR / 'bbdm_runs'
    OUTPUT_DRIVE_DIR = DRIVE_PROJECT_DIR / 'bbdm_outputs_standalone'

if DATA_SOURCE == 'huggingface':
    DATASET_HF_ID = HF_DATASET_ID
    print('Dataset source: Hugging Face')
    print('HF dataset:', DATASET_HF_ID)
elif DATA_SOURCE == 'drive':
    requested_dir = resolve_optional_path(DRIVE_DATASET_DIR)
    requested_zip = resolve_optional_path(DRIVE_DATASET_ZIP)

    if requested_dir is not None and requested_dir.exists():
        split_root = find_split_root(requested_dir)
        assert split_root is not None, f'No train/test split found under: {requested_dir}'
        DATASET_SOURCE_DIR = split_root
    elif requested_zip is not None:
        assert requested_zip.exists(), f'Dataset zip not found: {requested_zip}'
        DATASET_ZIP = requested_zip
    else:
        split_root = find_split_root(DRIVE_PROJECT_DIR)
        if split_root is not None:
            DATASET_SOURCE_DIR = split_root
        else:
            zip_candidates = sorted(
                DRIVE_PROJECT_DIR.rglob('*.zip'),
                key=lambda p: p.stat().st_size,
                reverse=True,
            )
            assert zip_candidates, f'No split directory or zip file found under: {DRIVE_PROJECT_DIR}'
            DATASET_ZIP = zip_candidates[0]
elif DATA_SOURCE == 'upload':
    from google.colab import files
    uploaded = files.upload()
    assert uploaded, 'No file uploaded.'
    DATASET_ZIP = Path(next(iter(uploaded.keys()))).resolve()
else:
    raise ValueError(f'Unknown DATA_SOURCE: {DATA_SOURCE}')

if DATASET_SOURCE_DIR is not None:
    print('Dataset split source:', DATASET_SOURCE_DIR)
elif DATASET_ZIP is not None:
    assert DATASET_ZIP.exists(), f'Dataset zip not found: {DATASET_ZIP}'
    print('Dataset zip:', DATASET_ZIP)
    print('Size MB:', DATASET_ZIP.stat().st_size / 1024 / 1024)

print('Drive project dir:', DRIVE_PROJECT_DIR if DRIVE_RUN_ROOT is not None else None)
print('Drive run root:', DRIVE_RUN_ROOT)
print('Output drive dir:', OUTPUT_DRIVE_DIR)


## 4. Prepare Dataset

Dữ liệu sẽ được đưa vào `/content/data`, sau đó notebook tự tìm split root chứa `train` và `test`.


In [ ]:
DATA_ROOT = Path('/content/data')
if DATA_ROOT.exists():
    shutil.rmtree(DATA_ROOT)
DATA_ROOT.mkdir(parents=True, exist_ok=True)


def save_hf_image(image, path, mode):
    image = image.convert(mode)
    path.parent.mkdir(parents=True, exist_ok=True)
    image.save(path)


def prepare_hf_split(dataset_split, split_name):
    input_dir = DATA_ROOT / split_name / 'input'
    target_dir = DATA_ROOT / split_name / 'target'
    input_dir.mkdir(parents=True, exist_ok=True)
    target_dir.mkdir(parents=True, exist_ok=True)

    for idx, item in enumerate(tqdm(dataset_split, desc=f'Preparing {split_name}')):
        filename = f'{idx:05d}.png'
        save_hf_image(item[HF_INPUT_COLUMN], input_dir / filename, 'L')
        save_hf_image(item[HF_TARGET_COLUMN], target_dir / filename, 'RGB')


if DATASET_HF_ID is not None:
    from datasets import load_dataset
    from tqdm.auto import tqdm

    hf_dataset = load_dataset(DATASET_HF_ID)
    for split_name in ['train', 'test']:
        assert split_name in hf_dataset, f'Hugging Face dataset missing split: {split_name}'
        prepare_hf_split(hf_dataset[split_name], split_name)

    if HF_EXPORT_PREPARED_ZIP_TO_DRIVE and DRIVE_RUN_ROOT is not None:
        archive = shutil.make_archive('/content/bach_dataset_hf_png', 'zip', DATA_ROOT)
        dest = DRIVE_PROJECT_DIR / 'data_hf_png.zip'
        shutil.copy2(archive, dest)
        print('Prepared Hugging Face PNG zip saved to:', dest)
elif DATASET_SOURCE_DIR is not None:
    shutil.copytree(DATASET_SOURCE_DIR, DATA_ROOT, dirs_exist_ok=True)
else:
    with zipfile.ZipFile(DATASET_ZIP) as zf:
        zf.extractall(DATA_ROOT)

SPLIT_ROOT = find_split_root(DATA_ROOT)
assert SPLIT_ROOT is not None, f'No train/test split found after preparing dataset under: {DATA_ROOT}'

TRAIN_INPUT_DIR = SPLIT_ROOT / 'train' / 'input'
TRAIN_TARGET_DIR = SPLIT_ROOT / 'train' / 'target'
TEST_INPUT_DIR = SPLIT_ROOT / 'test' / 'input'
TEST_TARGET_DIR = SPLIT_ROOT / 'test' / 'target'

for required_dir in [TRAIN_INPUT_DIR, TRAIN_TARGET_DIR, TEST_INPUT_DIR, TEST_TARGET_DIR]:
    assert required_dir.exists(), f'Required directory not found: {required_dir}'

print('Split root:', SPLIT_ROOT)
print('Train input dir:', TRAIN_INPUT_DIR)
print('Train target dir:', TRAIN_TARGET_DIR)
print('Test input dir:', TEST_INPUT_DIR)
print('Test target dir:', TEST_TARGET_DIR)


## 5. Inspect Dataset

Kỳ vọng:

- train paired count: 4800
- test paired count: 1200
- input mode `L`, size `(512, 512)`
- target mode `RGB`, size `(512, 512)`


In [ ]:
from PIL import Image

EXPECTED_TRAIN_PAIRS = 4800
EXPECTED_TEST_PAIRS = 1200


def image_files(directory):
    directory = Path(directory)
    files = []
    for pattern in ['*.png', '*.PNG', '*.jpg', '*.JPG', '*.jpeg', '*.JPEG']:
        files.extend(directory.glob(pattern))
    return sorted(files)


def inspect_split(split_name, input_dir, target_dir, expected_pairs):
    input_files = image_files(input_dir)
    target_files = image_files(target_dir)
    input_names = {p.name for p in input_files}
    target_names = {p.name for p in target_files}
    common_names = sorted(input_names & target_names)

    print(f'[{split_name}] input_count:', len(input_files))
    print(f'[{split_name}] target_count:', len(target_files))
    print(f'[{split_name}] paired_count:', len(common_names))
    print(f'[{split_name}] only_input:', len(input_names - target_names))
    print(f'[{split_name}] only_target:', len(target_names - input_names))
    assert len(common_names) == expected_pairs, f'{split_name} expected {expected_pairs} pairs, got {len(common_names)}'
    assert not (input_names - target_names), f'{split_name} has input files without target.'
    assert not (target_names - input_names), f'{split_name} has target files without input.'

    for name in common_names[:5]:
        inp = Image.open(Path(input_dir) / name)
        tgt = Image.open(Path(target_dir) / name)
        print(name, 'input', inp.mode, inp.size, '| target', tgt.mode, tgt.size)
    return common_names


train_names = inspect_split('train', TRAIN_INPUT_DIR, TRAIN_TARGET_DIR, EXPECTED_TRAIN_PAIRS)
test_names = inspect_split('test', TEST_INPUT_DIR, TEST_TARGET_DIR, EXPECTED_TEST_PAIRS)


## 6. Smoke Test Loader and Condition Encoder

Kiểm tra đường dữ liệu:

- train target batch: `[B, 3, 256, 256]`
- train condition batch: `[B, 1, 256, 256]`
- encoder output: `[B, 3, 256, 256]`

Ảnh condition gốc vẫn là grayscale; encoder output có thể là 3-channel màu vì đó là representation nội bộ cho diffusion.


In [ ]:
import matplotlib.pyplot as plt
import torch
from improved_diffusion.image_datasets import load_paired_png_data
from improved_diffusion.unet import ConditionEncoder

CROP_SIZE = 256
BATCH_SIZE = 1

preview_data = load_paired_png_data(
    input_dir=str(TRAIN_INPUT_DIR),
    target_dir=str(TRAIN_TARGET_DIR),
    batch_size=2,
    image_size=CROP_SIZE,
    deterministic=True,
)
target, cond, kwargs = next(preview_data)
print('target:', tuple(target.shape), float(target.min()), float(target.max()))
print('cond:', tuple(cond.shape), float(cond.mean()), float(cond.std()))

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(cond[0, 0].numpy(), cmap='gray')
axes[0].set_title('Raw grayscale condition')
axes[0].axis('off')
axes[1].imshow(((target[0].permute(1, 2, 0).numpy() + 1) * 127.5).clip(0, 255).astype('uint8'))
axes[1].set_title('RGB target')
axes[1].axis('off')
plt.tight_layout()
plt.show()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
encoder = ConditionEncoder(1, 3).to(device).eval()
with torch.no_grad():
    encoded = encoder(cond.to(device).float())
print('encoded:', tuple(encoded.shape), float(encoded.min()), float(encoded.max()))
assert target.shape[1:] == encoded.shape[1:], 'Target and encoded condition shapes must match.'


## 7. Optional Diffusion Loss Check

Mặc định tắt để tiết kiệm VRAM. Nếu muốn kiểm tra sâu hơn trước khi train, đặt `RUN_DIFFUSION_LOSS_CHECK = True`.


In [ ]:
RUN_DIFFUSION_LOSS_CHECK = False

if RUN_DIFFUSION_LOSS_CHECK:
    from improved_diffusion.script_util import sr_create_model_and_diffusion

    small_model, small_diffusion = sr_create_model_and_diffusion(
        large_size=CROP_SIZE,
        small_size=CROP_SIZE,
        class_cond=False,
        learn_sigma=False,
        num_channels=32,
        num_res_blocks=1,
        num_heads=4,
        num_heads_upsample=-1,
        attention_resolutions='16,8',
        dropout=0.0,
        diffusion_steps=10,
        noise_schedule='linear',
        timestep_respacing='',
        use_kl=False,
        predict_xstart=False,
        rescale_timesteps=True,
        rescale_learned_sigmas=True,
        use_checkpoint=False,
        use_scale_shift_norm=True,
        in_channels=3,
        out_channels=3,
    )
    small_model = small_model.to(device).eval()
    t = torch.randint(0, small_diffusion.num_timesteps, (target.shape[0],), device=device)
    with torch.no_grad():
        loss_terms = small_diffusion.training_losses(
            small_model,
            target.to(device).float(),
            encoded.float(),
            t,
        )
    print({k: float(v.mean().detach().cpu()) for k, v in loss_terms.items()})
else:
    print('Skipped diffusion loss check.')


## 8. Smoke Train

Chạy 20 step để kiểm tra toàn bộ training loop. Train dùng split `train`; validation/visualization dùng split `test`.


In [ ]:
import os
import subprocess
import sys

SMOKE_MODEL_DIR = Path('/content/bbdm_smoke_models')
SMOKE_LOG_DIR = Path('/content/bbdm_smoke_log')
for path in [SMOKE_MODEL_DIR, SMOKE_LOG_DIR]:
    if path.exists():
        shutil.rmtree(path)

cmd = [
    sys.executable, 'train.py.py',
    '--lr_data_dir', str(TRAIN_INPUT_DIR),
    '--hr_data_dir', str(TRAIN_TARGET_DIR),
    '--val_lr_data_dir', str(TEST_INPUT_DIR),
    '--val_hr_data_dir', str(TEST_TARGET_DIR),
    '--batch_size', str(BATCH_SIZE),
    '--large_size', str(CROP_SIZE),
    '--small_size', str(CROP_SIZE),
    '--num_channels', '32',
    '--num_res_blocks', '1',
    '--diffusion_steps', '20',
    '--lr_anneal_steps', '20',
    '--log_interval', '1',
    '--val_interval', '5',
    '--save_interval', '10',
    '--model_dir', str(SMOKE_MODEL_DIR),
    '--log_dir', str(SMOKE_LOG_DIR),
]

env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
env['DIFFUSION_DIST_BACKEND'] = 'gloo'

print(' '.join(cmd))
result = subprocess.run(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    env=env,
)
print(result.stdout)
if result.returncode != 0:
    raise RuntimeError(f'Smoke train failed with exit code {result.returncode}. See log above.')


## 9. Visualize Smoke Train Outputs

Hiển thị ảnh validation từ split `test`:

- Raw Grayscale Input
- Prediction
- Target

File `*_COND_ENCODED.png` vẫn được lưu trong log dir nếu cần debug encoder, nhưng không hiển thị mặc định vì nó là representation nội bộ 3-channel.


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image


def show_validation_triplets(log_dir, max_items=4):
    log_dir = Path(log_dir)
    triplets = []
    for idx in range(max_items):
        lr = log_dir / f'{idx}_LR.png'
        sr = log_dir / f'{idx}_SR.png'
        hr = log_dir / f'{idx}_HR.png'
        if lr.exists() and sr.exists() and hr.exists():
            triplets.append((lr, sr, hr))
    if not triplets:
        print('No validation images found in', log_dir)
        return

    fig, axes = plt.subplots(len(triplets), 3, figsize=(12, 4 * len(triplets)))
    if len(triplets) == 1:
        axes = [axes]
    for row, (lr, sr, hr) in zip(axes, triplets):
        for ax, path, title in zip(row, [lr, sr, hr], ['Raw Grayscale Input', 'Prediction', 'Target']):
            image = Image.open(path)
            if title == 'Raw Grayscale Input':
                ax.imshow(image, cmap='gray')
            else:
                ax.imshow(image)
            ax.set_title(title)
            ax.axis('off')
    plt.tight_layout()
    plt.show()


show_validation_triplets(SMOKE_LOG_DIR)


## 10. Longer Train

Sau khi smoke train chạy ổn, đổi `RUN_LONG_TRAIN = True` để train lâu hơn.

Train dùng split `train`; validation định kỳ dùng split `test`. Nếu `SAVE_CHECKPOINTS_TO_DRIVE = True`, checkpoint sẽ được ghi trực tiếp vào Drive để tránh mất khi Colab reset.


In [ ]:
RUN_LONG_TRAIN = True
LONG_STEPS = 5000
SAVE_CHECKPOINTS_TO_DRIVE = True
RESUME_CHECKPOINT = ''  # Example: '/content/drive/MyDrive/.../model005000.pt'
RUN_NAME = None  # Leave None to create a new timestamped run folder.

from datetime import datetime

print('Preparing long train cell...', flush=True)

if RESUME_CHECKPOINT:
    resume_path = Path(RESUME_CHECKPOINT)
    assert resume_path.exists(), f'Resume checkpoint not found: {resume_path}'
    LONG_MODEL_DIR = resume_path.parent
    if LONG_MODEL_DIR.name == 'models':
        LONG_RUN_ROOT = LONG_MODEL_DIR.parent
        LONG_LOG_DIR = LONG_RUN_ROOT / 'log'
    elif LONG_MODEL_DIR.name == 'bbdm_long_models':
        LONG_RUN_ROOT = LONG_MODEL_DIR.parent
        LONG_LOG_DIR = LONG_RUN_ROOT / 'bbdm_long_log'
    else:
        LONG_RUN_ROOT = LONG_MODEL_DIR.parent
        LONG_LOG_DIR = LONG_RUN_ROOT / 'log'
else:
    if RUN_NAME is None:
        RUN_NAME = 'bbdm_' + datetime.now().strftime('%Y%m%d_%H%M%S')
    if SAVE_CHECKPOINTS_TO_DRIVE and DRIVE_RUN_ROOT is not None:
        LONG_RUN_ROOT = DRIVE_RUN_ROOT / RUN_NAME
    else:
        LONG_RUN_ROOT = Path('/content') / RUN_NAME
    LONG_MODEL_DIR = LONG_RUN_ROOT / 'models'
    LONG_LOG_DIR = LONG_RUN_ROOT / 'log'

print('Run root:', LONG_RUN_ROOT, flush=True)
print('Model dir:', LONG_MODEL_DIR, flush=True)
print('Log dir:', LONG_LOG_DIR, flush=True)

if RUN_LONG_TRAIN:
    LONG_MODEL_DIR.mkdir(parents=True, exist_ok=True)
    LONG_LOG_DIR.mkdir(parents=True, exist_ok=True)
    print('Output directories ready.', flush=True)

    cmd = [
        sys.executable, 'train.py.py',
        '--lr_data_dir', str(TRAIN_INPUT_DIR),
        '--hr_data_dir', str(TRAIN_TARGET_DIR),
        '--val_lr_data_dir', str(TEST_INPUT_DIR),
        '--val_hr_data_dir', str(TEST_TARGET_DIR),
        '--batch_size', str(BATCH_SIZE),
        '--large_size', str(CROP_SIZE),
        '--small_size', str(CROP_SIZE),
        '--num_channels', '64',
        '--num_res_blocks', '2',
        '--diffusion_steps', '1000',
        '--lr_anneal_steps', str(LONG_STEPS),
        '--log_interval', '10',
        '--val_interval', '500',
        '--save_interval', '1000',
        '--model_dir', str(LONG_MODEL_DIR),
        '--log_dir', str(LONG_LOG_DIR),
    ]
    if RESUME_CHECKPOINT:
        cmd.extend(['--resume_checkpoint', RESUME_CHECKPOINT])

    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    env['DIFFUSION_DIST_BACKEND'] = 'gloo'
    print('Command:', ' '.join(cmd), flush=True)
    print('Starting training...', flush=True)

    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        env=env,
        bufsize=1,
    )
    for line in process.stdout:
        print(line, end='', flush=True)
    returncode = process.wait()
    if returncode != 0:
        raise RuntimeError(f'Long train failed with exit code {returncode}. See log above.')

    print('Long train finished.', flush=True)
    print('Long train model dir:', LONG_MODEL_DIR, flush=True)
    print('Long train log dir:', LONG_LOG_DIR, flush=True)
else:
    print('Long train skipped. Set RUN_LONG_TRAIN = True to run it.', flush=True)
    print('Long train model dir will be:', LONG_MODEL_DIR, flush=True)
    print('Long train log dir will be:', LONG_LOG_DIR, flush=True)


## 11. Save Outputs

Cell này gom checkpoint/log từ smoke train và long train rồi lưu về Drive hoặc tải zip về máy.


In [ ]:
SAVE_TO_DRIVE = True

export_root = Path('/content/bbdm_export')
if export_root.exists():
    shutil.rmtree(export_root)
export_root.mkdir(parents=True, exist_ok=True)

for src in [SMOKE_MODEL_DIR, SMOKE_LOG_DIR, LONG_MODEL_DIR, LONG_LOG_DIR]:
    src = Path(src)
    if src.exists():
        shutil.copytree(src, export_root / src.name)

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_DRIVE_DIR.parent.mkdir(parents=True, exist_ok=True)
    if OUTPUT_DRIVE_DIR.exists():
        shutil.rmtree(OUTPUT_DRIVE_DIR)
    shutil.copytree(export_root, OUTPUT_DRIVE_DIR)
    print('Saved to Drive:', OUTPUT_DRIVE_DIR)
else:
    from google.colab import files
    archive = shutil.make_archive('/content/bbdm_outputs_standalone', 'zip', export_root)
    files.download(archive)
    print('Prepared download:', archive)


## 12. Correct Inference Debug

Cell này thay cell inference thử nghiệm cũ. Điểm quan trọng là phải load cả checkpoint BBDM và checkpoint `*_compress.pt` của `ConditionEncoder`, đồng thời preprocess input grayscale giống training bằng z-score center crop.


In [ ]:
import re
import random
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from improved_diffusion.script_util import sr_create_model_and_diffusion
from improved_diffusion.unet import ConditionEncoder

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
INFER_SIZE = int(globals().get('CROP_SIZE', 256))
_default_model_dir = globals().get('LONG_MODEL_DIR', None)
if _default_model_dir is None:
    _default_model_dir = '/content/drive/MyDrive/bach_dataset_download/bbdm_runs/bbdm_20260516_071752/models'
INFER_MODEL_DIR = Path(_default_model_dir)
INFER_STEP = None  # None = latest checkpoint; or set 5000.
INFER_EMA_RATE = '0.9999'
INFER_TIMESTEP_RESPACING = ''  # Match training first. Try '250' only after exact sampling works.
INFER_NUM_SAMPLES_PER_SPLIT = 2
INFER_SPLITS = ('test', 'train')
INFER_FIXED_FILENAMES = []  # Example: ['00779.png', '00244.png', '00662.png']
INFER_COMPARE_RAW_AND_EMA = True
INFER_SHOW_PROGRESS = True

_STEP_RE = re.compile(r'(?:model|ema_[0-9.]+_)(\d{6})(?:_compress)?\.pt$')


def _torch_load_state_dict(path):
    try:
        return torch.load(path, map_location='cpu', weights_only=True)
    except TypeError:
        return torch.load(path, map_location='cpu')


def _checkpoint_step(path):
    match = _STEP_RE.search(Path(path).name)
    return int(match.group(1)) if match else -1


def _find_checkpoint_pair(model_dir, variant='ema', step=None):
    model_dir = Path(model_dir)
    if not model_dir.exists():
        raise FileNotFoundError(f'Model directory not found: {model_dir}')

    if variant == 'ema':
        pattern = f'ema_{INFER_EMA_RATE}_*.pt'
    elif variant == 'raw':
        pattern = 'model*.pt'
    else:
        raise ValueError(f'Unknown checkpoint variant: {variant}')

    candidates = [p for p in model_dir.glob(pattern) if '_compress' not in p.name and not p.name.startswith('opt')]
    if step is not None:
        candidates = [p for p in candidates if _checkpoint_step(p) == int(step)]
    if not candidates:
        raise FileNotFoundError(f'No {variant} checkpoint found in {model_dir} for step={step}')

    model_ckpt = max(candidates, key=_checkpoint_step)
    compress_ckpt = model_ckpt.with_name(model_ckpt.stem + '_compress.pt')
    if not compress_ckpt.exists():
        raise FileNotFoundError(f'Missing ConditionEncoder checkpoint: {compress_ckpt}')
    return model_ckpt, compress_ckpt


def _build_model_and_diffusion():
    return sr_create_model_and_diffusion(
        large_size=INFER_SIZE,
        small_size=INFER_SIZE,
        class_cond=False,
        learn_sigma=False,
        num_channels=64,
        num_res_blocks=2,
        num_heads=4,
        num_heads_upsample=-1,
        attention_resolutions='16,8',
        dropout=0.0,
        diffusion_steps=1000,
        noise_schedule='linear',
        timestep_respacing=INFER_TIMESTEP_RESPACING,
        use_kl=False,
        predict_xstart=False,
        rescale_timesteps=True,
        rescale_learned_sigmas=True,
        use_checkpoint=False,
        use_scale_shift_norm=True,
        in_channels=3,
        out_channels=3,
    )


def _load_checkpoint_variant(variant='ema'):
    model_ckpt, compress_ckpt = _find_checkpoint_pair(INFER_MODEL_DIR, variant=variant, step=INFER_STEP)
    model, diffusion = _build_model_and_diffusion()
    model.load_state_dict(_torch_load_state_dict(model_ckpt))
    model.to(DEVICE).eval()

    encoder = ConditionEncoder(1, 3).to(DEVICE).eval()
    encoder.load_state_dict(_torch_load_state_dict(compress_ckpt))
    encoder.to(DEVICE).eval()

    print(f'Loaded {variant} model:', model_ckpt)
    print(f'Loaded {variant} encoder:', compress_ckpt)
    return model, diffusion, encoder


def _center_crop_or_resize(image, size):
    width, height = image.size
    if width < size or height < size:
        return image.resize((size, size), Image.BICUBIC)
    left = (width - size) // 2
    top = (height - size) // 2
    return image.crop((left, top, left + size, top + size))


def _load_pair_for_inference(input_dir, target_dir, filename):
    raw_input = Image.open(Path(input_dir) / filename).convert('L')
    target = Image.open(Path(target_dir) / filename).convert('RGB')
    raw_input = _center_crop_or_resize(raw_input, INFER_SIZE)
    target = _center_crop_or_resize(target, INFER_SIZE)

    arr = np.asarray(raw_input, dtype=np.float32)
    arr = (arr - arr.mean()) / (arr.std() + 1e-6)
    tensor = torch.from_numpy(arr).unsqueeze(0).unsqueeze(0).to(DEVICE).float()
    return raw_input, target, tensor


def _tensor_to_uint8_image(tensor):
    array = tensor[0].detach().float().permute(1, 2, 0).cpu().numpy()
    return ((array + 1.0) * 127.5).clip(0, 255).astype(np.uint8)


def _split_dirs(split):
    if split == 'train':
        return (
            Path(globals().get('TRAIN_INPUT_DIR', '/content/data/train/input')),
            Path(globals().get('TRAIN_TARGET_DIR', '/content/data/train/target')),
        )
    if split == 'test':
        return (
            Path(globals().get('TEST_INPUT_DIR', '/content/data/test/input')),
            Path(globals().get('TEST_TARGET_DIR', '/content/data/test/target')),
        )
    raise ValueError(f'Unknown split: {split}')


def _select_filenames(input_dir, count):
    names = sorted(p.name for p in Path(input_dir).glob('*.png'))
    if INFER_FIXED_FILENAMES:
        selected = [name for name in INFER_FIXED_FILENAMES if name in names]
        if selected:
            return selected[:count]
    rng = random.Random(0)
    return rng.sample(names, min(count, len(names)))


def _sample_prediction(model, diffusion, encoder, lr_tensor):
    with torch.no_grad():
        cond_encoded = encoder(lr_tensor)
        sample = diffusion.p_sample_loop(
            model,
            cond_encoded,
            shape=(1, 3, INFER_SIZE, INFER_SIZE),
            clip_denoised=True,
            progress=INFER_SHOW_PROGRESS,
        )
    return sample, cond_encoded


def run_bbdm_inference_grid(variant='ema', split='test', num_samples=None):
    num_samples = INFER_NUM_SAMPLES_PER_SPLIT if num_samples is None else num_samples
    input_dir, target_dir = _split_dirs(split)
    filenames = _select_filenames(input_dir, num_samples)
    if not filenames:
        raise RuntimeError(f'No PNG files found in {input_dir}')

    model, diffusion, encoder = _load_checkpoint_variant(variant)
    fig, axes = plt.subplots(len(filenames), 4, figsize=(16, 4 * len(filenames)))
    if len(filenames) == 1:
        axes = [axes]

    for row, filename in zip(axes, filenames):
        raw_input, target, lr_tensor = _load_pair_for_inference(input_dir, target_dir, filename)
        sample, cond_encoded = _sample_prediction(model, diffusion, encoder, lr_tensor)
        prediction = _tensor_to_uint8_image(sample)
        encoded = _tensor_to_uint8_image(cond_encoded)

        print(
            f'{variant} | {split} | {filename} | '
            f'cond range=({cond_encoded.min().item():.3f}, {cond_encoded.max().item():.3f}) | '
            f'sample range=({sample.min().item():.3f}, {sample.max().item():.3f})'
        )

        panels = [raw_input, encoded, prediction, target]
        titles = [f'{split} input: {filename}', 'Loaded ConditionEncoder', f'{variant} prediction', 'Ground truth']
        for ax, image, title in zip(row, panels, titles):
            if title.startswith(f'{split} input'):
                ax.imshow(image, cmap='gray')
            else:
                ax.imshow(image)
            ax.set_title(title)
            ax.axis('off')

    plt.tight_layout()
    plt.show()


def compare_raw_vs_ema(split='test', filename=None):
    input_dir, target_dir = _split_dirs(split)
    if filename is None:
        filename = _select_filenames(input_dir, 1)[0]
    raw_input, target, lr_tensor = _load_pair_for_inference(input_dir, target_dir, filename)

    predictions = {}
    for variant in ('raw', 'ema'):
        try:
            model, diffusion, encoder = _load_checkpoint_variant(variant)
            sample, _ = _sample_prediction(model, diffusion, encoder, lr_tensor)
            predictions[variant] = _tensor_to_uint8_image(sample)
        except FileNotFoundError as exc:
            print(f'Skip {variant}: {exc}')

    columns = [('Input', raw_input), ('Ground truth', target)] + [(f'{k} prediction', v) for k, v in predictions.items()]
    fig, axes = plt.subplots(1, len(columns), figsize=(4 * len(columns), 4))
    if len(columns) == 1:
        axes = [axes]
    for ax, (title, image) in zip(axes, columns):
        if title == 'Input':
            ax.imshow(image, cmap='gray')
        else:
            ax.imshow(image)
        ax.set_title(f'{split} {filename} - {title}')
        ax.axis('off')
    plt.tight_layout()
    plt.show()


print('Inference model dir:', INFER_MODEL_DIR)
print('Device:', DEVICE)
if INFER_COMPARE_RAW_AND_EMA:
    compare_raw_vs_ema(split='test')
for split in INFER_SPLITS:
    run_bbdm_inference_grid(variant='ema', split=split)
